### Load libraries and data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')
from sklearn.cluster import KMeans
import plotly.express as px

url1 = "https://raw.githubusercontent.com/statzenthusiast921/wildfires/refs/heads/main/data/fire_df_wa.csv"
url2 = "https://raw.githubusercontent.com/statzenthusiast921/wildfires/refs/heads/main/data/fire_df_or.csv"
url3 = "https://raw.githubusercontent.com/statzenthusiast921/wildfires/refs/heads/main/data/fire_df_ca.csv"

df_wa = pd.read_csv(url1)
df_or = pd.read_csv(url2)
df_ca = pd.read_csv(url3, dtype={16: str, 18: str})


print(df_wa.shape)
print(df_or.shape)
print(df_ca.shape)

(33513, 19)
(61088, 19)
(189550, 19)


In [2]:
full_df = pd.concat([df_wa, df_or, df_ca], ignore_index=True)
full_df.shape

(284151, 19)

### Clean up data

In [3]:
full_df['EndDate'] = pd.to_datetime(full_df['CONT_DATE'], unit='D', origin='julian')
full_df['StartDate'] = pd.to_datetime(full_df['DISCOVERY_DATE'], unit='D', origin='julian')
full_df['FireLengthDays'] = full_df['CONT_DATE'] - full_df['DISCOVERY_DATE']

In [4]:
full_df = full_df[['LATITUDE','LONGITUDE','STATE','FireLengthDays','FIRE_YEAR','StartDate','FIRE_SIZE','STAT_CAUSE_DESCR']]

In [5]:
#full_df['MonthName'] = full_df['StartDate'].dt.strftime('%B')
full_df['MonthName'] = full_df['StartDate'].dt.month
full_df['month_sin'] = np.sin(2 * np.pi * full_df['MonthName'] / 12)
full_df['month_cos'] = np.cos(2 * np.pi * full_df['MonthName'] / 12)

In [6]:
full_df = full_df[~full_df['STAT_CAUSE_DESCR'].isin(['Structure','Railroad','Missing/Undefined'])]


In [7]:
full_df = pd.get_dummies(full_df, columns=['STAT_CAUSE_DESCR']).drop(columns=['STAT_CAUSE_DESCR_Fireworks'])

In [8]:
#----- Count fires per state per year
fires_per_year = full_df.groupby(['STATE', 'FIRE_YEAR']).size().reset_index(name='fire_count')

#----- Calculate average fires per year per state
avg_fires_per_state = fires_per_year.groupby('STATE')['fire_count'].mean()
avg_fires_per_state = avg_fires_per_state.to_dict()
avg_fires_per_state

{'CA': 7337.583333333333, 'OR': 2494.7916666666665, 'WA': 1355.5416666666667}

### Cluster Analysis

In [9]:
#----- Starting # of clusters
base_clusters = 15

#----- Compute proportional clusters
total_fires = sum(avg_fires_per_state.values())
clusters_per_state = {
    state: max(1, int(round(base_clusters * count / total_fires))) 
    for state, count in avg_fires_per_state.items()
}

#----- Initialize cluster column
full_df['location_cluster'] = -1

#----- Perform clustering per state with proportional clusters
for state, n_clusters in clusters_per_state.items():
    mask = full_df['STATE'] == state
    coords = full_df.loc[mask, ['LATITUDE', 'LONGITUDE']]
    
    if len(coords) >= n_clusters:
        kmeans = KMeans(n_clusters=n_clusters, random_state=42)
        clusters = kmeans.fit_predict(coords)
        full_df.loc[mask, 'location_cluster'] = clusters
    else:
        #----- Assign single cluster if not enough points
        full_df.loc[mask, 'location_cluster'] = 0


In [10]:
max_cluster = full_df['location_cluster'].max()

full_df.loc[(full_df['location_cluster'] == 0) & (full_df['STATE'] == 'WA'), 'location_cluster'] = max_cluster + 1
full_df.loc[(full_df['location_cluster'] == 0) & (full_df['STATE'] == 'OR'), 'location_cluster'] = max_cluster + 2
full_df.loc[(full_df['location_cluster'] == 1) & (full_df['STATE'] == 'WA'), 'location_cluster'] = max_cluster + 3
full_df.loc[(full_df['location_cluster'] == 1) & (full_df['STATE'] == 'OR'), 'location_cluster'] = max_cluster + 4
full_df.loc[(full_df['location_cluster'] == 2) & (full_df['STATE'] == 'OR'), 'location_cluster'] = max_cluster + 5

full_df['location_cluster'] = full_df['location_cluster'] + 1

### Put the dataset together for modelling

In [11]:
full_df = pd.get_dummies(full_df, columns=['location_cluster'], prefix='loc')

In [166]:
full_df['FIRE_SIZE_LOG'] = np.log1p(full_df['FIRE_SIZE'])

In [167]:
pred_cols = (
    [col for col in full_df.columns if col.startswith('STAT_CAUSE_DESCR_')] +  
    [col for col in full_df.columns if col.startswith('loc_')] +              
    ['month_sin', 'month_cos', 'FIRE_YEAR', 'FireLengthDays']      
)

response_col = 'FIRE_SIZE_LOG'

In [168]:
threshold = full_df['FIRE_SIZE_LOG'].quantile(0.75)
full_df['FIRE_SIZE_BUCKET'] = (full_df['FIRE_SIZE'] > threshold).astype(int)
full_df['FIRE_SIZE_BUCKET'].value_counts()

0    186131
1     82379
Name: FIRE_SIZE_BUCKET, dtype: int64

In [169]:
new_response_col = 'FIRE_SIZE_BUCKET'

In [170]:
#----- Sort by time
df_sorted = full_df.sort_values('StartDate')

#----- Define training data by cutting off at 75% threshold
cutoff = int(0.80 * len(df_sorted))
timestamp_cutoff = df_sorted['StartDate'].iloc[cutoff]

print("Cutoff at 80% of data:", timestamp_cutoff)

Cutoff at 80% of data: 2010-02-08 00:00:00


In [171]:
train_df = df_sorted[df_sorted['FIRE_YEAR'] < 2010]
test_df = df_sorted[df_sorted['FIRE_YEAR'] >= 2010]

### XGBoost #1

In [208]:
from xgboost import XGBClassifier

train_df = df_sorted[df_sorted['FIRE_YEAR'] < 2010]
test_df = df_sorted[df_sorted['FIRE_YEAR'] >= 2010]

#----- Create X and y for training
X_train = train_df[pred_cols]
y_train = train_df[new_response_col]

#----- Create X and y for testing
X_test = test_df[pred_cols]
y_test = test_df[new_response_col]

num_neg = np.sum(y_train == 0)  # small fires
num_pos = np.sum(y_train == 1)  # large fires

weight = round(num_neg / num_pos, 5)

clf = XGBClassifier(
    scale_pos_weight=weight,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

clf.fit(X_train, y_train);

In [232]:
from sklearn.metrics import classification_report, confusion_matrix

y_prob = clf.predict_proba(X_test)[:, 1]

#----- Apply custom threshold
threshold = 0.75  #increase to favor small-fire predictions
y_pred_adjusted = (y_prob >= threshold).astype(int)

#----- Evaluate
print(classification_report(y_test, y_pred_adjusted))
print(confusion_matrix(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

           0       0.75      0.98      0.85     30653
           1       0.65      0.09      0.15     10719

    accuracy                           0.75     41372
   macro avg       0.70      0.54      0.50     41372
weighted avg       0.73      0.75      0.67     41372

[[30157   496]
 [ 9793   926]]


### Results weren't great so let's over sample large fires a bit

In [214]:
from imblearn.over_sampling import RandomOverSampler

# Oversample large fires to half the size of the small-fire class
minority_class_size = sum(y_train == 1)
target_large_fire_count = max(minority_class_size + 1, int(majority_class_size * 0.1))

ros = RandomOverSampler(sampling_strategy={1: target_large_fire_count}, random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

In [235]:
clf_res = XGBClassifier(
    scale_pos_weight=weight,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)
clf_res.fit(X_resampled, y_resampled);#added in semicolon to suppress annoying output

In [236]:
y_prob = clf_res.predict_proba(X_test)[:, 1]

#----- Apply custom threshold
threshold = 0.75 
y_pred_adjusted = (y_prob >= threshold).astype(int)

#----- Evaluate
print(classification_report(y_test, y_pred_adjusted))
print(confusion_matrix(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

           0       0.75      0.98      0.85     30653
           1       0.65      0.09      0.15     10719

    accuracy                           0.75     41372
   macro avg       0.70      0.54      0.50     41372
weighted avg       0.73      0.75      0.67     41372

[[30157   496]
 [ 9793   926]]


### That didn't work too well, let's try something else - RF?

In [237]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)

X_train = X_train.dropna()
y_train = y_train[X_train.index]
X_test = X_test.dropna()
y_test = y_test[X_test.index]

rf.fit(X_train, y_train)

#----- Apply custom threshold
threshold = 0.75 
y_prob = rf.predict_proba(X_test)[:, 1]
y_pred_adjusted = (y_prob >= threshold).astype(int)

#----- Evaluate
print(classification_report(y_test, y_pred_adjusted))
print(confusion_matrix(y_test, y_pred_adjusted))

              precision    recall  f1-score   support

           0       0.75      0.99      0.85     30653
           1       0.67      0.08      0.14     10719

    accuracy                           0.75     41372
   macro avg       0.71      0.53      0.50     41372
weighted avg       0.73      0.75      0.67     41372

[[30232   421]
 [ 9856   863]]


### Goal is to focus more on small fires since they're more common and large fires are a bonus